# Data quality, missing values & outlier treatment

Companion to **`EDA.ipynb`**. This notebook narrates **what** `src/preprocessing.py` implements and **why**, with visual before/after checks on `data/raw/texas_houston_raw.csv` → `data/processed/texas_houston_clean.csv`.

**Base-project parallel:** Gurgaon workflow split *missing_value_handling*, *outlier_handling*, *preprocessing* — here we document the same decisions for the Texas pipeline.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", 60)
PROJECT_ROOT = Path("..").resolve()
raw = pd.read_csv(PROJECT_ROOT / "data/raw/texas_houston_raw.csv", low_memory=False)
clean = pd.read_csv(PROJECT_ROOT / "data/processed/texas_houston_clean.csv", low_memory=False)
print(raw.shape, "->", clean.shape)


## 1. Missing values — strategy

| Column group | Strategy in `preprocessing.py` |
|--------------|----------------------------------|
| Numeric (`bedrooms`, `bathrooms`, `sqft`, …) | `median` imputation after coercion |
| `property_type`, `location` | fill unknown / strip strings |
| `description` | empty string |
| Metadata (`address`, `listing_url`, …) | preserved when present |

Below: missing rates on **raw** input.


In [ ]:
mr = raw.isna().mean().sort_values(ascending=False)
display(mr[mr > 0].head(25).to_frame("missing_rate"))


## 2. Outlier removal — IQR on core numerics

The cleaner applies IQR with factor 1.5 on `price`, `sqft`, `bedrooms`, `bathrooms` **after** basic price bounds. We visualize impact on price.


In [ ]:
fig = px.histogram(raw, x="price", nbins=80, title="RAW: price histogram")
fig.update_layout(template="plotly_white")
fig.show()
fig2 = px.histogram(clean, x="price", nbins=80, title="CLEAN: price histogram")
fig2.update_layout(template="plotly_white")
fig2.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(raw["price"].dropna(), label="raw", fill=True, alpha=0.35)
sns.kdeplot(clean["price"].dropna(), label="clean", fill=True, alpha=0.35)
plt.title("Price density: raw vs clean")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Texas / Houston filter

Rows kept if `state` is TX (or similar) **or** `location` contains Houston (case-insensitive), matching `preprocessing.py` intent for regional focus.


In [ ]:
# Quick sanity: state distribution in clean
if "state" in clean.columns:
    display(clean["state"].value_counts().head(10))


## 4. Missingness after cleaning

Core numerics should be filled post–median impute; sparse URL fields may remain null by design.


In [ ]:
core = [c for c in ["price", "sqft", "bedrooms", "bathrooms", "year_built", "latitude", "longitude"] if c in clean.columns]
display(clean[core].isna().mean().to_frame("missing_rate_after_clean"))


## 5. Reproduce from CLI

`python src/preprocessing.py`

Tune IQR columns, multiplier, or price bounds in `src/preprocessing.py` if the business requires retaining luxury tails.
